# EYES-DEFY-ANEMIA -- Phase 4 Classification (v2, CLEAN DATA) -- Transformer Architectures

3 transformer architectures x 2 tissue types = 6 combos (`swin_t`, `vit_b_16`, `vit_l_16`), retrained against the reprocessed dataset after the white-background bug fix (2026-08-01). Queued for Kaggle's "Save Version -> Save & Run All" background execution, cheapest architecture first -- `vit_l_16` (304.3M params) is deliberately queued last, so a session truncation loses the single most expensive combo, not an earlier cheap one.

The 6 CNN architectures are deliberately **not** included here -- run in a separate session instead (`classification-cnn-clean.ipynb`). Named "Transformer", not strictly "ViT", because `swin_t` (Swin Transformer, local-attention) is grouped here too, matching this project's existing `architecture_family` categorization -- not literally the ViT architecture family (`vit_b_16`/`vit_l_16` only), which would leave `swin_t` with nowhere to go in a two-notebook split.

## Setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first (this bit an earlier
# version of this notebook during interactive debugging).
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

Cloning into 'eyes-defy-anemia'...
remote: Enumerating objects: 652, done.
remote: Counting objects: 100% (333/333), done.
remote: Compressing objects: 100% (225/225), done.
remote: Total 652 (delta 143), reused 289 (delta 107), pack-reused 319 (from 1)
Receiving objects: 100% (652/652), 67.41 MiB | 26.93 MiB/s, done.
Resolving deltas: 100% (306/306), done.
/kaggle/working/eyes-defy-anemia


In [3]:
# Diagnostic only -- kept for visibility in the saved run log. The path used
# below is already confirmed correct, so this cell doesn't gate anything, but
# it's cheap and makes a future path change easy to spot in the output.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

datasets -> ['manivafapour33']


In [4]:
# Only packages actually missing from Kaggle's base image. Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
!pip install -q optuna albumentations

## Data

In [5]:
import shutil
from pathlib import Path

# TODO: verify against cell 4's /kaggle/input listing before running -- this is a best-guess
# default following the established manivafapour33/<slug> pattern, NOT yet confirmed for the
# new clean-data dataset (see classification/.project_memory/kaggle/01_kaggle_notes.md).
SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

# Clear the destination first so this cell is fully idempotent -- a re-run (or
# the two messy copy attempts from the earlier notebook version) never leaves
# stale or duplicated content behind. DST_DIR always ends up as an exact,
# deterministic copy of SRC_DIR, nothing more.
shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

classification/data/processed now contains:
  extraction_log.csv
  images/  (428 files)
  metadata.csv
  splits.csv


In [6]:
import sys

sys.path.insert(0, "classification/datapreparepipeline")
from dataset import get_dataloaders

loaders = get_dataloaders("palpebral")
images, labels, countries = next(iter(loaders["train"]))
print("Batch shape:", images.shape)
print("Train patients:", len(loaders["train"].dataset))
print("Val patients:", len(loaders["val"].dataset))

Batch shape: torch.Size([32, 3, 256, 256])
Train patients: 151
Val patients: 33


## Training -- 6 combos, clean data, cheapest architecture first

Retrains this subset of the 18-combo v2 sweep against the reprocessed dataset (white-background bug fixed, `classification/.project_memory/02_current_status.md` "Data bug fixed" entry, 2026-08-01). Each script's `model_name` carries a `_v2_clean` suffix (not a bare `_v2` re-run) specifically so these results never silently overwrite the original v2 dirty-data results under the same filename -- both stay on disk, comparable side-by-side. Same v2 protocol otherwise (100-epoch ceiling, patience=7, dropout_rate tuned, 12-trial Optuna search) -- nothing about the search itself changed, only the input images.

In [7]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate classification/outputs/{checkpoints,logs,plots}/ into a
    single top-level /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/vit_clean_results.zip. Called after EVERY training cell
    below, not just at the end -- if the run gets cut short partway
    through the 6 combos (a real possibility on a long unattended Save &
    Run All), whatever completed so far is still cleanly consolidated and
    zipped, ready to download, rather than only existing nested several
    directories deep with no single downloadable archive."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("classification/outputs") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive("/kaggle/working/vit_clean_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

[sync_outputs] 0 files consolidated under /kaggle/working/outputs, zipped to /kaggle/working/vit_clean_results.zip


In [8]:
# Clean 1/6 -- Swin-Tiny, palpebral
!python classification/v2_clean_scripts/train_swin_t_palpebral_v2_clean.py
sync_outputs()

Using device: cuda
Architecture: swin_t
Tissue type: palpebral
Model name: swin_t_palpebral_v2_clean
[I 2026-08-02 15:46:18,744] A new study created in memory with name: no-name-cda1565d-96c1-49cc-a4ce-0c4e3aff1bf2
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth
100%|█████████████████████████████████████████| 108M/108M [00:00<00:00, 176MB/s]
[swin_t_palpebral_v2_clean | Trial 0] New best overall val_f1=0.6222 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_swin_t_palpebral_v2_clean.pth
[swin_t_palpebral_v2_clean | Trial 0] Epoch  1/250 - train_loss=0.8389 val_loss=0.7721 val_acc=0.4848 val_f1=0.6222 India_acc=0.7857 Italy_acc=0.2632
[swin_t_palpebral_v2_clean | Trial 0] New best overall val_f1=0.7000 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_swin_t_palpebral_v2_clean.pth
[swin_t_palpebral_v2_clean | Trial 0] Epoch  2/250 - train_loss

In [9]:
# Clean 2/6 -- Swin-Tiny, forniceal_palpebral
!python classification/v2_clean_scripts/train_swin_t_forniceal_palpebral_v2_clean.py
sync_outputs()

Using device: cuda
Architecture: swin_t
Tissue type: forniceal_palpebral
Model name: swin_t_forniceal_palpebral_v2_clean
[I 2026-08-02 16:01:15,105] A new study created in memory with name: no-name-4814999f-6c14-4e5f-a615-7103b36075f7
[swin_t_forniceal_palpebral_v2_clean | Trial 0] New best overall val_f1=0.5556 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_swin_t_forniceal_palpebral_v2_clean.pth
[swin_t_forniceal_palpebral_v2_clean | Trial 0] Epoch  1/250 - train_loss=0.8211 val_loss=0.8125 val_acc=0.4839 val_f1=0.5556 India_acc=0.4286 Italy_acc=0.5294
[swin_t_forniceal_palpebral_v2_clean | Trial 0] Epoch  2/250 - train_loss=0.7657 val_loss=0.7932 val_acc=0.5161 val_f1=0.3478 India_acc=0.3571 Italy_acc=0.6471
[swin_t_forniceal_palpebral_v2_clean | Trial 0] Epoch  3/250 - train_loss=0.7623 val_loss=0.7752 val_acc=0.5806 val_f1=0.5185 India_acc=0.5000 Italy_acc=0.6471
[swin_t_forniceal_palpebral_v2_clean | Trial 0] New best overall val_f1=0.6667 -> sa

In [10]:
# Clean 3/6 -- ViT-B/16, palpebral
!python classification/v2_clean_scripts/train_vit_b_16_palpebral_v2_clean.py
sync_outputs()

Using device: cuda
Architecture: vit_b_16
Tissue type: palpebral
Model name: vit_b_16_palpebral_v2_clean
[I 2026-08-02 16:13:25,099] A new study created in memory with name: no-name-67e39a4e-3aa9-4b91-8a34-aed637f3f50d
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|█████████████████████████████████████████| 330M/330M [00:01<00:00, 240MB/s]
[vit_b_16_palpebral_v2_clean | Trial 0] New best overall val_f1=0.6400 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_vit_b_16_palpebral_v2_clean.pth
[vit_b_16_palpebral_v2_clean | Trial 0] Epoch  1/250 - train_loss=0.7905 val_loss=0.7260 val_acc=0.7273 val_f1=0.6400 India_acc=0.7143 Italy_acc=0.7368
[vit_b_16_palpebral_v2_clean | Trial 0] New best overall val_f1=0.8000 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_vit_b_16_palpebral_v2_clean.pth
[vit_b_16_palpebral_v2_clean | Trial 0] Epoch

In [11]:
# Clean 4/6 -- ViT-B/16, forniceal_palpebral
!python classification/v2_clean_scripts/train_vit_b_16_forniceal_palpebral_v2_clean.py
sync_outputs()

Using device: cuda
Architecture: vit_b_16
Tissue type: forniceal_palpebral
Model name: vit_b_16_forniceal_palpebral_v2_clean
[I 2026-08-02 17:09:34,185] A new study created in memory with name: no-name-0ce2838d-7fd8-4591-b539-6af81ecc918a
[vit_b_16_forniceal_palpebral_v2_clean | Trial 0] New best overall val_f1=0.3810 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_vit_b_16_forniceal_palpebral_v2_clean.pth
[vit_b_16_forniceal_palpebral_v2_clean | Trial 0] Epoch  1/250 - train_loss=0.7788 val_loss=0.7740 val_acc=0.5806 val_f1=0.3810 India_acc=0.5000 Italy_acc=0.6471
[vit_b_16_forniceal_palpebral_v2_clean | Trial 0] New best overall val_f1=0.6061 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_vit_b_16_forniceal_palpebral_v2_clean.pth
[vit_b_16_forniceal_palpebral_v2_clean | Trial 0] Epoch  2/250 - train_loss=0.7116 val_loss=0.7310 val_acc=0.5806 val_f1=0.6061 India_acc=0.5714 Italy_acc=0.5882
[vit_b_16_forniceal_palpebr

In [12]:
# Clean 5/6 -- ViT-L/16, palpebral
!python classification/v2_clean_scripts/train_vit_l_16_palpebral_v2_clean.py
sync_outputs()

Using device: cuda
Architecture: vit_l_16
Tissue type: palpebral
Model name: vit_l_16_palpebral_v2_clean
[I 2026-08-02 17:38:20,908] A new study created in memory with name: no-name-7e81117e-14b7-4769-b9db-5f92d5d121d3
Downloading: "https://download.pytorch.org/models/vit_l_16_lc_swag-4d563306.pth" to /root/.cache/torch/hub/checkpoints/vit_l_16_lc_swag-4d563306.pth
100%|███████████████████████████████████████| 1.13G/1.13G [00:07<00:00, 156MB/s]
[vit_l_16_palpebral_v2_clean | Trial 0] New best overall val_f1=0.5806 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_vit_l_16_palpebral_v2_clean.pth
[vit_l_16_palpebral_v2_clean | Trial 0] Epoch  1/250 - train_loss=0.8075 val_loss=0.7426 val_acc=0.6061 val_f1=0.5806 India_acc=0.5000 Italy_acc=0.6842
[vit_l_16_palpebral_v2_clean | Trial 0] New best overall val_f1=0.6667 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_vit_l_16_palpebral_v2_clean.pth
[vit_l_16_palpebral_v2_clean 

In [13]:
# Clean 6/6 -- ViT-L/16, forniceal_palpebral
!python classification/v2_clean_scripts/train_vit_l_16_forniceal_palpebral_v2_clean.py
sync_outputs()

Using device: cuda
Architecture: vit_l_16
Tissue type: forniceal_palpebral
Model name: vit_l_16_forniceal_palpebral_v2_clean
[I 2026-08-02 19:07:45,789] A new study created in memory with name: no-name-828314e2-2479-4a5f-acad-8250370a8b0d
[vit_l_16_forniceal_palpebral_v2_clean | Trial 0] New best overall val_f1=0.4348 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_vit_l_16_forniceal_palpebral_v2_clean.pth
[vit_l_16_forniceal_palpebral_v2_clean | Trial 0] Epoch  1/250 - train_loss=0.8842 val_loss=0.7911 val_acc=0.5806 val_f1=0.4348 India_acc=0.4286 Italy_acc=0.7059
[vit_l_16_forniceal_palpebral_v2_clean | Trial 0] Epoch  2/250 - train_loss=0.7678 val_loss=0.7861 val_acc=0.5484 val_f1=0.2222 India_acc=0.3571 Italy_acc=0.7059
[vit_l_16_forniceal_palpebral_v2_clean | Trial 0] New best overall val_f1=0.7586 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_vit_l_16_forniceal_palpebral_v2_clean.pth
[vit_l_16_forniceal_palpebr

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` (checkpoints, logs, plots for whichever combos completed) and zipped to `/kaggle/working/vit_clean_results.zip`. Both are visible in this notebook version's **Output** tab once you Save Version -> Save & Run All -- download the zip directly from there, or browse the folder for individual files.

In [14]:
print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.2f} MB)')

zip_path = Path('/kaggle/working/vit_clean_results.zip')
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")

Final contents of /kaggle/working/outputs:
  checkpoints/best_swin_t_forniceal_palpebral_v2_clean.pth  (110.38 MB)
  checkpoints/best_swin_t_palpebral_v2_clean.pth  (110.38 MB)
  checkpoints/best_vit_b_16_forniceal_palpebral_v2_clean.pth  (343.26 MB)
  checkpoints/best_vit_b_16_palpebral_v2_clean.pth  (343.26 MB)
  checkpoints/best_vit_l_16_forniceal_palpebral_v2_clean.pth  (1213.34 MB)
  checkpoints/best_vit_l_16_palpebral_v2_clean.pth  (1213.34 MB)
  logs/swin_t_forniceal_palpebral_v2_clean_study_summary.json  (0.00 MB)
  logs/swin_t_forniceal_palpebral_v2_clean_trials.csv  (0.12 MB)
  logs/swin_t_palpebral_v2_clean_study_summary.json  (0.00 MB)
  logs/swin_t_palpebral_v2_clean_trials.csv  (0.14 MB)
  logs/vit_b_16_forniceal_palpebral_v2_clean_study_summary.json  (0.00 MB)
  logs/vit_b_16_forniceal_palpebral_v2_clean_trials.csv  (0.11 MB)
  logs/vit_b_16_palpebral_v2_clean_study_summary.json  (0.00 MB)
  logs/vit_b_16_palpebral_v2_clean_trials.csv  (0.19 MB)
  logs/vit_l_16_forniceal